# 002: Tool Executor — The Shared Execution Pipeline

This notebook walks through everything in spec 002: how arcrun dispatches tool calls.

Before 002, the tool execution pipeline was inline in `react.py` (~50 lines of sandbox check, registry lookup, validation, execute, error handling). Spec 002 extracted it into `executor.py` — a single shared function that any strategy can call.

By the end you'll understand:

1. **The execution pipeline** — the 10-step process every tool call goes through
2. **Sandbox permissions** — deny-by-default, allowlist, custom checkers
3. **Schema validation** — JSON Schema enforced before execute()
4. **Error handling** — exceptions caught, truncated for LLM, full in events
5. **Timeout** — per-tool and global timeout with asyncio.wait_for
6. **Events** — tool.start, tool.end, tool.error, tool.denied
7. **How react_loop uses it** — control flow stays in the strategy

Everything runs with mocks — no API keys needed.

In [ ]:
# Setup: add project paths
import os
import sys

sys.path.insert(0, os.path.abspath("../src"))
sys.path.insert(0, os.path.abspath("../tests"))

---

## 1. The Tool — What the Model Can Call

A `Tool` is a dataclass with 5 fields. The model sees `name`, `description`, and `input_schema`. The executor calls `execute`.

In [ ]:
from arcrun.types import Tool


# Define a simple tool
async def search_execute(params, ctx):
    query = params["query"]
    # In a real tool, this would query a database or API
    return f"Found 3 results for '{query}': [result1, result2, result3]"


search_tool = Tool(
    name="search",
    description="Search the knowledge base",
    input_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"},
        },
        "required": ["query"],
    },
    execute=search_execute,
)

print(f"name:            {search_tool.name}")
print(f"description:     {search_tool.description}")
print(f"input_schema:    {search_tool.input_schema}")
print(f"timeout_seconds: {search_tool.timeout_seconds}  (None = use global or no timeout)")
print(f"execute:         {search_tool.execute}")

### The execute signature

```python
async def execute(params: dict[str, Any], ctx: ToolContext) -> str
```

- `params` — already validated against `input_schema` before your code runs
- `ctx` — provides `run_id`, `tool_call_id`, `turn_number`, `event_bus`, and `cancelled` signal
- Returns a string (goes back to the model)
- Raise an exception for errors (executor catches it)

In [ ]:
import asyncio

from arcrun.types import ToolContext

# ToolContext gives the tool environment awareness
ctx = ToolContext(
    run_id="demo-001",
    tool_call_id="tc-42",
    turn_number=3,
    event_bus=None,  # can emit custom events
    cancelled=asyncio.Event(),  # check if execution should stop
)

print(f"run_id:       {ctx.run_id}")
print(f"tool_call_id: {ctx.tool_call_id}")
print(f"turn_number:  {ctx.turn_number}")
print(f"cancelled:    {ctx.cancelled.is_set()}")

---

## 2. The Execution Pipeline — 10 Steps

`execute_tool_call` is the shared pipeline. Every tool call — from every strategy — goes through these exact 10 steps:

```
execute_tool_call(tc, state, sandbox)
  │
  ├── 1. Emit tool.start event
  ├── 2. Sandbox.check() — permission check
  │       denied? → return error, False
  ├── 3. Registry.get() — look up tool definition
  │       not found? → return error, False
  ├── 4. jsonschema.validate() — validate arguments
  │       invalid? → return error, False
  ├── 5. Build ToolContext
  ├── 6. Execute tool (with optional timeout)
  ├── 7. Catch exceptions → emit tool.error, return error, False
  ├── 8. Emit tool.end event (with duration_ms)
  ├── 9. Increment state.tool_calls_made
  └── 10. Return (tool_result_message, True)
```

Let's see each step in action.

In [ ]:
from conftest import Message, ToolCall

from arcrun.events import EventBus
from arcrun.executor import execute_tool_call
from arcrun.registry import ToolRegistry
from arcrun.sandbox import Sandbox
from arcrun.state import RunState


# Helper: set up the infrastructure for a tool call
def setup(tools=None, sandbox_config=None, tool_timeout=None):
    if tools is None:
        tools = [search_tool]
    bus = EventBus(run_id="demo")
    state = RunState(
        messages=[Message(role="user", content="go")],
        registry=ToolRegistry(tools=tools, event_bus=bus),
        event_bus=bus,
        run_id="demo",
        tool_timeout=tool_timeout,
    )
    sandbox = Sandbox(config=sandbox_config, event_bus=bus)
    return state, sandbox, bus

### 2a. Happy Path — All 10 Steps Succeed

In [ ]:
state, sandbox, bus = setup()

# Simulate a tool call from the model
tc = ToolCall(id="tc-1", name="search", arguments={"query": "arcrun architecture"})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Tool calls tracked: {state.tool_calls_made}")
print(f"Result message role: {result_msg.role}")
print(f"Result content: {result_msg.content[0].content}")
print()
print("Events emitted:")
for event in bus.events:
    print(f"  {event.type:15s} {event.data}")

Notice:
- `tool.start` fires first (with the tool name and arguments)
- `tool.end` fires after success (with `result_length` and `duration_ms`)
- `state.tool_calls_made` incremented to 1
- The result message is in arcllm format — role="tool" with a `ToolResultBlock`

### 2b. Sandbox Denied — Step 2 Fails

In [ ]:
from arcrun.types import SandboxConfig

# Sandbox only allows "read_file" — our "search" tool is NOT in the list
state, sandbox, bus = setup(sandbox_config=SandboxConfig(allowed_tools=["read_file"]))
tc = ToolCall(id="tc-2", name="search", arguments={"query": "secret data"})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Tool calls tracked: {state.tool_calls_made}  (not incremented)")
print(f"Error sent to model: {result_msg.content[0].content}")
print()
print("Events:")
for event in bus.events:
    print(f"  {event.type:15s} {event.data}")

The sandbox denied the call. The model gets an error message (so it can adjust), a `tool.denied` event fires, and `tool_calls_made` stays at 0.

### 2c. Custom Sandbox Checker

In [ ]:
# Custom checker: allow search, but deny queries containing "password"
async def policy_checker(tool_name, params):
    if tool_name == "search" and "password" in params.get("query", "").lower():
        return False, "queries about passwords are not allowed"
    return True, ""


config = SandboxConfig(
    allowed_tools=["search"],  # search is in the allowlist
    check=policy_checker,  # BUT custom checker adds granular control
)

# Allowed query
state, sandbox, bus = setup(sandbox_config=config)
tc = ToolCall(id="tc-ok", name="search", arguments={"query": "architecture"})
_, ok = await execute_tool_call(tc, state, sandbox)
print(f"'architecture' query: ok={ok}")

# Denied query
state, sandbox, bus = setup(sandbox_config=config)
tc = ToolCall(id="tc-bad", name="search", arguments={"query": "admin password"})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"'admin password' query: ok={ok}")
print(f"  reason: {result_msg.content[0].content}")

### 2d. Tool Not Found — Step 3 Fails

In [ ]:
state, sandbox, bus = setup()
tc = ToolCall(id="tc-3", name="nonexistent_tool", arguments={})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Error sent to model: {result_msg.content[0].content}")

### 2e. Schema Validation Failure — Step 4 Fails

In [ ]:
# search requires "query" (type: string). Let's pass an integer.
state, sandbox, bus = setup()
tc = ToolCall(id="tc-4", name="search", arguments={"query": 42})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Error: {result_msg.content[0].content}")

In [ ]:
# Missing required field
state, sandbox, bus = setup()
tc = ToolCall(id="tc-5", name="search", arguments={})  # no "query" field

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Error: {result_msg.content[0].content}")

Schema validation happens *before* `execute()` is called. Your tool code never sees invalid arguments.

### 2f. Tool Exception — Step 7 Catches It

In [ ]:
# A tool that raises an exception
async def broken_execute(params, ctx):
    raise ConnectionError("database connection refused")


broken_tool = Tool(
    name="broken",
    description="Always fails",
    input_schema={"type": "object"},
    execute=broken_execute,
)

state, sandbox, bus = setup(tools=[broken_tool])
tc = ToolCall(id="tc-6", name="broken", arguments={})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Tool calls tracked: {state.tool_calls_made}  (not incremented on failure)")
print(f"Error to model: {result_msg.content[0].content}")
print()
print("Events:")
for event in bus.events:
    print(f"  {event.type:15s} {event.data}")

The executor catches the exception, emits `tool.error` with the full error, and sends a truncated version back to the model. The loop continues — the model can adjust its approach.

### 2g. Error Truncation

Tool errors sent to the model are truncated to 200 characters (controlled by `_MAX_ERROR_LEN`). The event log gets the full error.

In [ ]:
async def verbose_error(params, ctx):
    raise RuntimeError("x" * 500)  # 500-char error message


verbose_tool = Tool(
    name="verbose",
    description="Verbose errors",
    input_schema={"type": "object"},
    execute=verbose_error,
)

state, sandbox, bus = setup(tools=[verbose_tool])
tc = ToolCall(id="tc-7", name="verbose", arguments={})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

llm_error = result_msg.content[0].content
event_error = next(e for e in bus.events if e.type == "tool.error").data["error"]

print(f"LLM sees {len(llm_error)} chars (truncated)")
print(f"Event log has {len(event_error)} chars (full)")
print()
print(f"LLM error: {llm_error[:80]}...")
print(f"Event error ends with: ...{event_error[-20:]}")

Why truncate for the LLM? Long error messages waste context tokens. The model only needs enough to understand *what* failed. The full error is always in the event log for debugging.

---

## 3. Timeout — Per-Tool and Global

Tools can specify a `timeout_seconds`. If not set, the global `state.tool_timeout` is used. If neither is set, no timeout.

In [ ]:
import asyncio


async def slow_execute(params, ctx):
    await asyncio.sleep(10)  # takes 10 seconds
    return "done"


# Per-tool timeout: 0.1 seconds
slow_tool = Tool(
    name="slow",
    description="Takes forever",
    input_schema={"type": "object"},
    execute=slow_execute,
    timeout_seconds=0.1,  # ← per-tool timeout
)

state, sandbox, bus = setup(tools=[slow_tool])
tc = ToolCall(id="tc-slow", name="slow", arguments={})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Error: {result_msg.content[0].content}")
print()
print("Events:")
for event in bus.events:
    print(f"  {event.type:15s} {event.data}")

In [ ]:
# Global timeout: set on state, applies to tools without their own timeout
no_timeout_tool = Tool(
    name="slow",
    description="Takes forever",
    input_schema={"type": "object"},
    execute=slow_execute,
    # No timeout_seconds set
)

state, sandbox, bus = setup(tools=[no_timeout_tool], tool_timeout=0.1)
tc = ToolCall(id="tc-slow2", name="slow", arguments={})

result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"Success: {ok}")
print(f"Used global timeout: {result_msg.content[0].content}")

Timeout priority: `tool.timeout_seconds` > `state.tool_timeout` > no timeout.

This is implemented in `executor.py` line 56:
```python
timeout = tool_def.timeout_seconds or state.tool_timeout
```

---

## 4. The ToolRegistry — Dynamic Tool Collection

The executor looks up tools in the `ToolRegistry`. Tools can be added, removed, or replaced mid-execution.

In [ ]:
bus = EventBus(run_id="registry-demo")
registry = ToolRegistry(tools=[search_tool], event_bus=bus)

print(f"Tools: {registry.names()}")
print(f"Get 'search': {registry.get('search').name}")
print(f"Get 'missing': {registry.get('missing')}")

In [ ]:
# Add a tool dynamically
async def write_execute(params, ctx):
    return f"Wrote to {params['path']}"


write_tool = Tool(
    name="write_file",
    description="Write to a file",
    input_schema={
        "type": "object",
        "properties": {"path": {"type": "string"}},
        "required": ["path"],
    },
    execute=write_execute,
)

registry.add(write_tool)
print(f"After add: {registry.names()}")

# Remove a tool
registry.remove("search")
print(f"After remove: {registry.names()}")

# Events emitted for registry changes
print("\nRegistry events:")
for event in bus.events:
    print(f"  {event.type}: {event.data}")

In [ ]:
# list_schemas() converts tools to arcllm format for model.invoke()
bus = EventBus(run_id="schema-demo")
registry = ToolRegistry(tools=[search_tool], event_bus=bus)

schemas = registry.list_schemas()
print(f"Schema count: {len(schemas)}")
print(f"Type: {type(schemas[0]).__name__}")
print(f"  name: {schemas[0].name}")
print(f"  description: {schemas[0].description}")
print(f"  parameters: {schemas[0].parameters}")

The loop calls `state.registry.list_schemas()` every turn. So if you add or remove tools mid-execution, the model sees the updated list on its next turn.

---

## 5. The Sandbox — Permission Boundary

The sandbox sits between the model's intent and tool execution. It checks permissions before every call.

In [ ]:
from arcrun.types import SandboxConfig

bus = EventBus(run_id="sandbox-demo")

# No config = no sandbox = all allowed
sandbox = Sandbox(config=None, event_bus=bus)
allowed, reason = await sandbox.check("anything", {})
print(f"No sandbox: allowed={allowed}, reason={reason!r}")

In [ ]:
# Allowlist: only these tools can run
bus = EventBus(run_id="sandbox-allowlist")
sandbox = Sandbox(
    config=SandboxConfig(allowed_tools=["search", "read_file"]),
    event_bus=bus,
)

allowed, _ = await sandbox.check("search", {})
print(f"'search':     allowed={allowed}")

allowed, reason = await sandbox.check("delete_db", {})
print(f"'delete_db':  allowed={allowed}, reason={reason!r}")

# tool.denied event was emitted
denied = [e for e in bus.events if e.type == "tool.denied"]
print(f"\nDenied events: {len(denied)}")
print(f"  {denied[0].data}")

In [ ]:
# Custom checker: fine-grained control based on arguments
async def file_checker(tool_name, params):
    path = params.get("path", "")
    if "/etc/" in path or "/root/" in path:
        return False, f"access to {path} denied"
    if path.endswith(".env"):
        return False, "cannot read .env files"
    return True, ""


bus = EventBus(run_id="sandbox-checker")
sandbox = Sandbox(
    config=SandboxConfig(allowed_tools=["read_file"], check=file_checker),
    event_bus=bus,
)

# Safe path
allowed, _ = await sandbox.check("read_file", {"path": "/data/report.txt"})
print(f"/data/report.txt: allowed={allowed}")

# Dangerous path
allowed, reason = await sandbox.check("read_file", {"path": "/etc/shadow"})
print(f"/etc/shadow:      allowed={allowed}, reason={reason!r}")

# .env file
allowed, reason = await sandbox.check("read_file", {"path": "/app/.env"})
print(f"/app/.env:        allowed={allowed}, reason={reason!r}")

In [ ]:
# Fail-safe: checker exceptions are treated as denial
async def buggy_checker(tool_name, params):
    raise RuntimeError("checker crashed")


bus = EventBus(run_id="sandbox-failsafe")
sandbox = Sandbox(
    config=SandboxConfig(allowed_tools=["search"], check=buggy_checker),
    event_bus=bus,
)

allowed, reason = await sandbox.check("search", {})
print(f"Buggy checker: allowed={allowed}, reason={reason!r}")
print("(Fail-safe: exception = denial)")

---

## 6. The Event System

Every action emits an event. Events are collected on the bus and optionally forwarded to a handler.

In [ ]:
from arcrun.events import EventBus


# EventBus with a real-time handler
def my_handler(event):
    print(f"  >> {event.type}: {event.data}")


bus = EventBus(run_id="event-demo", on_event=my_handler)

print("Emitting events:")
bus.emit("demo.start", {"message": "hello"})
bus.emit("demo.end", {"duration": 42})

print(f"\nCollected events: {len(bus.events)}")
for event in bus.events:
    print(f"  type={event.type}, run_id={event.run_id}, data={event.data}")

In [ ]:
# Handler exceptions don't crash the engine
def broken_handler(event):
    raise RuntimeError("handler crashed!")


bus = EventBus(run_id="safe", on_event=broken_handler)
event = bus.emit("test", {"data": "still works"})
print(f"Event emitted despite handler crash: {event.type}")
print(f"Events collected: {len(bus.events)}")

### Tool-related events

The executor emits these events:

| Event | When | Data |
|---|---|---|
| `tool.start` | Before any checks | `{name, arguments}` |
| `tool.end` | After successful execution | `{name, result_length, duration_ms}` |
| `tool.error` | Tool raised exception or timed out | `{name, error}` |
| `tool.denied` | Sandbox rejected the call | `{name, reason}` |

The sandbox emits `tool.denied`. The executor emits the rest.

---

## 7. How react_loop Uses the Executor

The key insight of spec 002: **the executor handles execution, the strategy handles control flow**.

Here's the relevant section from `react.py` (lines 107-127):

```python
# Process tool calls
tool_results: list[Any] = []
steered = False
for tc in response.tool_calls:
    if steered or state.cancel_event.is_set():
        tool_results.append(tool_result(tc.id, "operation cancelled: steered"))
        continue

    result_msg, _ok = await execute_tool_call(tc, state, sandbox)
    tool_results.append(result_msg)

    if not state.steer_queue.empty():
        steer_msg = state.steer_queue.get_nowait()
        state.messages.append(user_message(steer_msg))
        steered = True

for tr in tool_results:
    state.messages.append(tr)
```

The strategy (react_loop) handles:
- Cancel/steer checks (before calling executor)
- Collecting results into a list
- Appending results to messages
- Turn counting

The executor handles:
- Sandbox check
- Registry lookup
- Schema validation
- Execution with timing
- Error handling
- Event emission

This separation means future strategies (CodeExec, Recursive) get the same execution pipeline without copying code.

In [ ]:
# Full end-to-end through run() to see everything working together
from conftest import LLMResponse, MockModel

from arcrun.loop import run

model = MockModel(
    [
        # Turn 1: model calls search
        LLMResponse(
            content="Let me search for that.",
            tool_calls=[ToolCall(id="tc1", name="search", arguments={"query": "arcrun design"})],
            stop_reason="tool_use",
        ),
        # Turn 2: model gives answer
        LLMResponse(
            content="Based on the search results, arcrun is an execution engine.",
            stop_reason="end_turn",
        ),
    ]
)

events_log = []


def collector(event):
    events_log.append(event)


result = await run(
    model=model,
    tools=[search_tool],
    system_prompt="You are a helpful assistant.",
    task="Tell me about arcrun's design",
    on_event=collector,
)

print(f"Content:  {result.content}")
print(f"Turns:    {result.turns}")
print(f"Tools:    {result.tool_calls_made}")
print(f"Strategy: {result.strategy_used}")
print(f"Events:   {len(result.events)}")
print()
print("Full event trail:")
for event in result.events:
    print(f"  {event.type}")

---

## Summary

Spec 002 extracted 50 lines of inline tool execution from `react.py` into a shared `executor.py`.

### The 10-Step Pipeline

```
1. tool.start event          ──►  audit trail
2. sandbox.check()           ──►  permission boundary
3. registry.get()            ──►  tool lookup
4. jsonschema.validate()     ──►  argument validation
5. build ToolContext          ──►  execution environment
6. tool.execute()            ──►  run the tool
7. catch exceptions          ──►  error handling
8. tool.end event            ──►  audit trail + duration
9. increment tool_calls_made ──►  tracking
10. return (message, bool)   ──►  result for strategy
```

### What the executor does NOT do

| Concern | Who handles it |
|---|---|
| Cancel/steer checks | Strategy (react_loop) |
| Message appending | Strategy |
| Turn counting | Strategy |
| Loop control flow | Strategy |

### Why this matters

One function. One pipeline. Every strategy uses it. When we add timeout, truncation, rate limiting, or output sanitization — we add it once, in `executor.py`, and every strategy gets it for free.